In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/noc_regions.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/olympics-data.xlsx
/kaggle/input/datasets/lukajincharadze/pandas-data/results.parquet
/kaggle/input/datasets/lukajincharadze/pandas-data/bios.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/coffee.csv
/kaggle/input/datasets/lukajincharadze/pandas-data/results.csv


In [2]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 59.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

# DagsHub/Mlflow Initialization

In [3]:
import dagshub
dagshub.init(repo_owner='skoba23', repo_name='ML_Assignment_2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=d2a4ca28-88c7-468a-b279-416de7c885f5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=99afcbbf176db340773dbc9a3d957f7766e136aa769090d8ccc881e64bdf0522




Output()

Accessing as skoba23

Initialized MLflow to track repo "skoba23/ML_Assignment_2"

Repository skoba23/ML_Assignment_2 initialized!

In [4]:
import mlflow
import mlflow.sklearn
import os
TRACKING_URI = "https://dagshub.com/skoba23/ML_Assignment_2.mlflow"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("LogisticRegression_Training")

print('MLflow Tracking URI:', TRACKING_URI)
print('Setup complete!')

MLflow Tracking URI: https://dagshub.com/skoba23/ML_Assignment_2.mlflow
Setup complete!


## Data Loading

In [5]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,   on='TransactionID', how='left')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print('Fraud rate: ', round(train['isFraud'].mean() * 100, 2), '%')

Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  3.5 %


# Cleaning

In [1]:
with mlflow.start_run(run_name = "LR_Cleaning"):
    null_threshold = 0.6
    null_ratio = train.isnull().mean()
    #dropping columns
    drop_cols = null_ratio[null_ratio > null_threshold].index.tolist()
    drop_cols = [c for c in drop_cols if c != "isFraud"]
    
    df_train_clean = train.drop(columns=drop_cols)
    df_test_clean  = test.drop(columns=[c for c in drop_cols if c in test.columns])

    #Filling null values with median
    num_cols = df_train_clean.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c != "isFraud"]
    df_train_clean[num_cols] = df_train_clean[num_cols].fillna(df_train_clean[num_cols].median())
    df_test_clean[num_cols] = df_test_clean[[c for c in num_cols if c in df_test_clean.columns]].fillna(df_train_clean[[c for c in num_cols if c in df_test_clean.columns]].median())

    #One-Hot Encoding
    from sklearn.preprocessing import LabelEncoder
    cat_cols = df_train_clean.select_dtypes(include="object").columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        df_train_clean[col] = le.fit_transform(df_train_clean[col].astype(str))
        if col in df_test_clean.columns:
            df_test_clean[col] = le.transform(df_test_clean[col].astype(str).map(
                lambda x: x if x in le.classes_ else le.classes_[0]))
            
    mlflow.log_param("null_threshold", null_threshold)
    mlflow.log_param("encoding", "LabelEncoder")
    mlflow.log_param("dropped_columns", len(drop_cols))
    mlflow.log_param("remaining_features", df_train_clean.shape[1])
    
    print(f"Dropped {len(drop_cols)} columns, remaining: {df_train_clean.shape[1]}")

mlflow.end_run()

NameError: name 'mlflow' is not defined

# Feature Engineering

In [11]:
with mlflow.start_run(run_name = "LR_FeatureEngineering"):
    df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
    df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
    
    df_train_clean["Transaction_day"]  = (df_train_clean["TransactionDT"] // (3600 * 24)) % 7
    df_train_clean["Transaction_hour"] = (df_train_clean["TransactionDT"] // 3600) % 24
    df_test_clean["Transaction_day"]   = (df_test_clean["TransactionDT"] // (3600 * 24)) % 7
    df_test_clean["Transaction_hour"]  = (df_test_clean["TransactionDT"] // 3600) % 24
    
    mlflow.log_param("new_features", "TransactionAmt_log, Transaction_day, Transaction_hour")
    mlflow.log_metric("total_features", df_train_clean.shape[1])
    
    print("Feature Engineering done: ", df_train_clean.shape)

/tmp/ipykernel_57/1978296550.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_train_clean["TransactionAmt_log"] = np.log1p(df_train_clean["TransactionAmt"])
/tmp/ipykernel_57/1978296550.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_test_clean["TransactionAmt_log"]  = np.log1p(df_test_clean["TransactionAmt"])
/tmp/ipykernel_57/1978296550.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

Feature Engineering done:  (590540, 229)
🏃 View run LR_FeatureEngineering at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/cba786deb1a2444589116e1ffcf88249
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


# Feature Selection

In [12]:
from sklearn.feature_selection import SelectKBest, f_classif

Y = df_train_clean["isFraud"]
X = df_train_clean.drop(columns=["isFraud", "TransactionID"])

with mlflow.start_run(run_name = "LR_FeatureSelection"):
    selector = SelectKBest(f_classif, k=50)
    X_selected = selector.fit_transform(X, Y)
    selected_features = X.columns[selector.get_support()].tolist()
    
    mlflow.log_param("method", "SelectKBest_f_classif")
    mlflow.log_param("k", 50)
    mlflow.log_metric("selected_features", len(selected_features))
    
    print(f"Selected {len(selected_features)} features")
    print(selected_features[:10])

X_final = X[selected_features]
X_test_final = df_test_clean[[c for c in selected_features if c in df_test_clean.columns]]

Selected 50 features
['ProductCD', 'card3', 'V15', 'V16', 'V17', 'V18', 'V21', 'V22', 'V23', 'V31']
🏃 View run LR_FeatureSelection at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/d2024cd2073f45628453cb0611bb2925
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


### filling missing columns with 0

In [13]:
for col in selected_features:
    if col not in X_test_final.columns:
        X_test_final[col] = 0
X_test_final = X_test_final[selected_features]

# Training

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import mlflow.sklearn

X_tr, X_val, Y_tr, Y_val = train_test_split(X_final, Y, test_size=0.2, random_state=42, stratify=Y)

### Baseline

In [15]:
with mlflow.start_run(run_name = "LR_Train_Baseline"):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, solver="lbfgs", max_iter=100, random_state=42))
    ])
    pipe.fit(X_tr, Y_tr)
    preds = pipe.predict_proba(X_val)[:,1]
    auc = roc_auc_score(Y_val, preds)

    mlflow.log_params({"C": 1.0, "solver": "lbfgs", "max_iter": 100})
    mlflow.log_metric("val_auc", auc)
    mlflow.sklearn.log_model(pipe, "model")
    print(f"Baseline AUC: {auc:.4f}")

2026/05/07 08:34:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:34:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline AUC: 0.7232
🏃 View run LR_Train_Baseline at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/a6bbd7ab655942d38efb8e0ef6472606
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


### Tuning C

In [16]:
with mlflow.start_run(run_name="LR_Train_TuningC"):
    pipe2 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=10.0, solver="lbfgs", max_iter=200, random_state=42))
    ])
    pipe2.fit(X_tr, Y_tr)
    preds2 = pipe2.predict_proba(X_val)[:, 1]
    auc2 = roc_auc_score(Y_val, preds2)

    mlflow.log_params({"C": 10.0, "solver": "lbfgs", "max_iter": 200})
    mlflow.log_metric("val_auc", auc2)
    mlflow.sklearn.log_model(pipe2, "model")
    print(f"Tuning C AUC: {auc2:.4f}")

2026/05/07 08:34:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:34:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Tuning C AUC: 0.7232
🏃 View run LR_Train_TuningC at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/b7920e8b88c9493db76a682e9635f0dc
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


### Class Weight Balanced

In [17]:
with mlflow.start_run(run_name="LR_Train_BalancedClasses"):
    pipe3 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, solver="lbfgs", max_iter=200,
                                     class_weight="balanced", random_state=42))
    ])
    pipe3.fit(X_tr, Y_tr)
    preds3 = pipe3.predict_proba(X_val)[:, 1]
    auc3 = roc_auc_score(Y_val, preds3)

    mlflow.log_params({"C": 1.0, "solver": "lbfgs", "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc3)
    mlflow.sklearn.log_model(pipe3, "model")
    print(f"Balanced AUC: {auc3:.4f}")

2026/05/07 08:34:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:34:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Balanced AUC: 0.7289
🏃 View run LR_Train_BalancedClasses at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/0772ada8b6a74fe2a8a1da9b39361166
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


In [18]:
with mlflow.start_run(run_name="LR_Train_BalancedClasses_2"):
    pipe4 = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, solver="saga", penalty = "l1",
                                     class_weight="balanced", max_iter=100))
    ])
    pipe4.fit(X_tr, Y_tr)
    preds4 = pipe4.predict_proba(X_val)[:, 1]
    auc4 = roc_auc_score(Y_val, preds4)

    mlflow.log_params({"C": 1.0, "solver": "lbfgs", "class_weight": "balanced"})
    mlflow.log_metric("val_auc", auc4)
    mlflow.sklearn.log_model(pipe4, "model")
    print(f"Balanced AUC: {auc4:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2026/05/07 08:36:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:36:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Balanced AUC: 0.7287
🏃 View run LR_Train_BalancedClasses_2 at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/b0aef3ea29594623aaee7cde52bf9dd8
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


### Cross Validation

In [19]:
with mlflow.start_run(run_name="LR_C1_balanced_CV5"):
    cv_scores = cross_val_score(pipe3, X_final, Y, cv=5, scoring="roc_auc")

    mlflow.log_param("cv_folds", 5)
    mlflow.log_metric("cv_auc_mean", cv_scores.mean())
    mlflow.log_metric("cv_auc_std", cv_scores.std())
    print(f"CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

CV AUC: 0.7155 ± 0.0345
🏃 View run LR_C1_balanced_CV5 at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/b5437111ba5d46d59c36de51793a05d9
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0


# Saving to Model Registry

In [20]:
with mlflow.start_run(run_name="LR_BestModel_Registry"):
    mlflow.sklearn.log_model(
        pipe3,
        artifact_path="model",
        registered_model_name="IEEE_Fraud_LogisticRegression"
    )
    mlflow.log_metric("val_auc", auc3)
    print("Best model saved to Registry!")

AUC results: {'pipe': np.float64(0.723151276218403), 'pipe2': np.float64(0.7231527155295462), 'pipe3': np.float64(0.728875580096259)}


2026/05/07 08:37:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 08:37:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best model saved to Registry!
🏃 View run LR_BestModel_Registry at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0/runs/5fe73170fe4944578ac98d72e860f0a0
🧪 View experiment at: https://dagshub.com/skoba23/ML_Assignment_2.mlflow/#/experiments/0
